# 17 - Configure ECG-focused CXR-augmented report generation

The primary question is whether CXR-derived context retrieved through a bridge learned from
same-patient Symile observations improves **ECG machine-style diagnosis generation**.

Retrieval method and context source are separated. This is necessary because a condition that mixes
ECG and CXR neighbors cannot identify the specific contribution of CXR knowledge.

| Context scope | Purpose |
|---|---|
| `zero_shot` | ECG waveform only |
| `ecg_only_rag` | active ECG-to-ECG neighbor RAG control (ECG-ReGen-style, not an exact reproduction) |
| `cxr_only_rag` | isolates CXR augmentation |
| `ecg_plus_cxr_rag` | tests complementary ECG and CXR context |
| `random_cxr_rag` | controls for extra medical text and prompt length |

The ECG-only conditions are the critical same-modality retrieval baseline identified by the IEEE ECG-RAG
literature: they retrieve external training ECG machine diagnoses for the query ECG. CLIP-style continuous,
CodeBind shared-codebook, and reciprocal-rank-fused retrieval use the same ECG
cohort and case budget. Retrieved records are always training records from other patients. Same-patient
concurrence is bridge-training supervision; an external retrieved CXR is not assumed to belong to the
ECG query patient or to share its diagnosis.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
BASE_CONFIG_PATH = ARTIFACT_ROOT / "config.json"
if not BASE_CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Run notebooks 00 and 09 first; missing {BASE_CONFIG_PATH}")
base_config = json.loads(BASE_CONFIG_PATH.read_text())

RAG_ROOT = ARTIFACT_ROOT / "rag_report_generation_ecg"
for relative in ["contexts", "query_images/ecg", "generations", "reports", "logs"]:
    (RAG_ROOT / relative).mkdir(parents=True, exist_ok=True)

MODELS = {
    "medgemma_4b": os.environ.get(
        "RAG_MEDGEMMA_MODEL", "google/medgemma-1.5-4b-it"),
    "qwen35_4b": os.environ.get("RAG_QWEN_MODEL", "Qwen/Qwen3.5-4B"),
}
RAG_SEED = int(os.environ.get("RAG_SEED", "5107"))
MAX_TEST_ECG = int(os.environ.get("RAG_MAX_TEST_ECG", os.environ.get("RAG_MAX_TEST_PER_MODALITY", "250")))
CONTEXT_CASES = int(os.environ.get("RAG_CONTEXT_CASES", "4"))
BOTH_ECG_CASES = int(os.environ.get("RAG_BOTH_ECG_CASES", "2"))
BOTH_CXR_CASES = int(os.environ.get("RAG_BOTH_CXR_CASES", "2"))
MAX_CONTEXT_CHARACTERS = int(os.environ.get("RAG_MAX_CONTEXT_CHARACTERS", "6000"))
MAX_NEW_TOKENS = int(os.environ.get("RAG_MAX_NEW_TOKENS", "384"))
CODEBIND_VARIANT = os.environ.get("RAG_CODEBIND_VARIANT", "shared_specific_vq")
CXR_CONTEXT_CONTENT = os.environ.get(
    "RAG_CXR_CONTEXT_CONTENT", "labels_and_report").strip().lower()
if MAX_TEST_ECG < 1 or CONTEXT_CASES < 1 or MAX_CONTEXT_CHARACTERS < 1:
    raise ValueError("ECG cohort, context-case, and character budgets must be positive")
if BOTH_ECG_CASES < 1 or BOTH_CXR_CASES < 1 or BOTH_ECG_CASES + BOTH_CXR_CASES != CONTEXT_CASES:
    raise ValueError("RAG_BOTH_ECG_CASES + RAG_BOTH_CXR_CASES must equal RAG_CONTEXT_CASES")
if CXR_CONTEXT_CONTENT not in {"report", "labels", "labels_and_report"}:
    raise ValueError("RAG_CXR_CONTEXT_CONTENT must be report, labels, or labels_and_report")


In [ ]:
condition_specs = [{
    "condition": "zero_shot", "retrieval_method": "none", "context_scope": "none",
    "ecg_cases": 0, "cxr_cases": 0,
}]
for method in ["clip", "codebind", "combined"]:
    condition_specs.extend([
        {"condition": f"{method}_ecg_only_rag", "retrieval_method": method,
         "context_scope": "ecg_only", "ecg_cases": CONTEXT_CASES, "cxr_cases": 0},
        {"condition": f"{method}_cxr_only_rag", "retrieval_method": method,
         "context_scope": "cxr_only", "ecg_cases": 0, "cxr_cases": CONTEXT_CASES},
        {"condition": f"{method}_ecg_plus_cxr_rag", "retrieval_method": method,
         "context_scope": "ecg_plus_cxr", "ecg_cases": BOTH_ECG_CASES,
         "cxr_cases": BOTH_CXR_CASES},
    ])
condition_specs.append({
    "condition": "random_cxr_rag", "retrieval_method": "random",
    "context_scope": "cxr_only", "ecg_cases": 0, "cxr_cases": CONTEXT_CASES,
})
CONDITIONS = [row["condition"] for row in condition_specs]
if len(CONDITIONS) != len(set(CONDITIONS)):
    raise AssertionError("Duplicate RAG condition names")
print(json.dumps(condition_specs, indent=2))


In [ ]:
experiment_path = ARTIFACT_ROOT / "ablation" / "experiment_index.json"
experiments = json.loads(experiment_path.read_text())
if not isinstance(experiments, list) or not experiments:
    raise ValueError(f"No bridge experiments in {experiment_path}; run notebook 12")

def select_by_validation(variant):
    candidates = [row for row in experiments if row.get("variant") == variant]
    candidates = [row for row in candidates if row.get("best_validation_retrieval_r1") is not None]
    if not candidates:
        raise KeyError(f"No validation-scored bridge variant {variant!r}")
    return max(candidates, key=lambda row: (
        float(row["best_validation_retrieval_r1"]), -int(row["seed"])))

selected = {
    "clip": select_by_validation("clip_continuous"),
    "codebind": select_by_validation(CODEBIND_VARIANT),
}
if selected["codebind"]["variant"] in {"clip_continuous", "shuffled_pairs_control"}:
    raise ValueError("RAG_CODEBIND_VARIANT must be a non-control VQ bridge")
print("Validation-selected bridge runs:", json.dumps(selected, indent=2))


In [ ]:
required_files = []
for modality in ["ecg", "cxr"]:
    required_files.extend([
        ARTIFACT_ROOT / "external" / modality / "manifest.jsonl",
        ARTIFACT_ROOT / "external" / modality / "concepts.jsonl",
        ARTIFACT_ROOT / "external" / modality / "encoding_status.npy",
    ])
    if modality == "cxr":
        required_files.append(ARTIFACT_ROOT / "external" / modality / "manifest_summary.json")
    for experiment in selected.values():
        required_files.append(
            ARTIFACT_ROOT / "external" / modality / "bridge" /
            experiment["variant"] / f"seed_{experiment['seed']}" / "common.npy")
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "ECG-focused RAG prerequisites are missing. Run notebooks 10-14:\n" +
        "\n".join(missing[:40]))

cxr_summary = json.loads(
    (ARTIFACT_ROOT / "external" / "cxr" / "manifest_summary.json").read_text())
expected_count = int(os.environ.get(
    "MIMIC_CXR_EXPECTED_TRAIN_IMAGES",
    base_config.get("external_cxr_expected_train_images", 43700)))
expected_roots = {
    "provided_train": str(Path(os.environ.get(
        "MIMIC_CXR_TRAIN_ROOT", base_config.get("external_cxr_root", "")))),
    "provided_test": str(Path(os.environ.get(
        "MIMIC_CXR_TEST_ROOT", base_config.get("external_cxr_test_root", "")))),
}
observed_count = int(cxr_summary.get("discovery", {}).get("provided_train", {}).get("images", -1))
observed_roots = cxr_summary.get("roots", {})
if observed_count != expected_count or observed_roots != expected_roots:
    raise RuntimeError(
        "Stale external CXR artifacts. "
        f"Expected {expected_count:,} train images and {expected_roots}; "
        f"found {observed_count:,} and {observed_roots}. Rerun notebooks 09, 11, 13, and 14.")

rag_config = {
    "artifact_root": str(ARTIFACT_ROOT), "rag_root": str(RAG_ROOT),
    "primary_modality": "ecg", "models": MODELS,
    "conditions": CONDITIONS, "condition_specs": condition_specs,
    "seed": RAG_SEED, "max_test_ecg": MAX_TEST_ECG,
    "context_cases": CONTEXT_CASES, "both_ecg_cases": BOTH_ECG_CASES,
    "both_cxr_cases": BOTH_CXR_CASES,
    "max_context_characters": MAX_CONTEXT_CHARACTERS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "cxr_context_content": CXR_CONTEXT_CONTENT,
    "retrieval_selection": selected,
    "hf_cache": base_config.get("hf_cache"),
    "generation_protocol_version": "ecg_cxr_scope_ablation_v1",
    "primary_comparisons": [
        "ECG-to-ECG neighbor RAG versus zero-shot",
        "retrieved CXR versus zero-shot", "ECG+CXR versus ECG-only",
        "CodeBind CXR versus CLIP CXR", "retrieved CXR versus random CXR",
    ],
    "evaluation_text_metrics": [
        "BLEU-1", "BLEU-4", "METEOR", "ROUGE-1/2/L", "BERTScore-F1", "CIDEr",
    ],
    "ecg_neighbor_baseline_note": (
        "ECG-only RAG is an ECG-ReGen-style same-modality control, not an exact reproduction; "
        "it uses this study's validation-selected bridge embeddings and external ECG machine diagnoses."),
    "leakage_controls": [
        "queries are external ECG split=test",
        "all retrieval candidates have split=train",
        "query sample IDs are forbidden",
        "same subject IDs are excluded when available",
        "query machine diagnosis and labels are never included in prompts",
        "bridge selection uses validation retrieval only",
    ],
}
config_path = RAG_ROOT / "rag_config.json"
config_path.write_text(json.dumps(rag_config, indent=2))
print("Saved:", config_path)


In [ ]:
required_packages = [
    "numpy", "pandas", "torch", "transformers", "accelerate", "PIL", "wfdb",
    "matplotlib", "sklearn", "nltk", "rouge_score", "bert_score", "pycocoevalcap",
]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
print({"missing_packages": missing_packages})
if missing_packages:
    print("Install missing packages in the Biowulf environment:")
    print("pip install -U 'transformers>=5.14.1' accelerate nltk rouge-score bert-score pycocoevalcap scikit-learn")
try:
    from importlib.metadata import version as package_version
    print("Transformers:", package_version("transformers"),
          "(the working MedGemma 1.5 reference notebook used 5.14.1)")
except Exception as exc:
    print("Could not read the Transformers version:", repr(exc))
print("MedGemma 1.5 requires accepted Hugging Face terms and authentication on Biowulf.")
